In [ ]:
# --- Librerias ---
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # evita el choque de OpenMP (torch + cv2)
import time, csv
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

In [ ]:
# --- Parametros y rutas ---
tipo_modelo = "vit_h"
ruta_checkpoint = r"vit_h.pth"
ruta_imagen = r"datos/escritorio.jpg"
carpeta_salida = r"resultados"
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(carpeta_salida, exist_ok=True)

In [ ]:
# --- Cargar la imagen ---
imagen_rgb = cv2.cvtColor(cv2.imread(ruta_imagen), cv2.COLOR_BGR2RGB)
alto, ancho = imagen_rgb.shape[:2]
pixeles_totales = alto * ancho

In [ ]:
# --- Cargar SAM y generar las mascaras ---
modelo_sam = sam_model_registry[tipo_modelo](checkpoint=ruta_checkpoint).to(dispositivo)
generador = SamAutomaticMaskGenerator(modelo_sam)
inicio = time.perf_counter()
mascaras = generador.generate(imagen_rgb)
print(f"{len(mascaras)} mascaras en {time.perf_counter() - inicio:.1f} s ({dispositivo})")

In [ ]:
# --- Metricas de las mascaras ---
union = np.zeros((alto, ancho), dtype=bool)
solape = np.zeros((alto, ancho), dtype=bool)
areas, ious, estabilidades = [], [], []
for m in mascaras:
    seg = m["segmentation"]
    areas.append(int(seg.sum()))
    ious.append(float(m["predicted_iou"]))
    estabilidades.append(float(m["stability_score"]))
    solape |= union & seg
    union |= seg
cobertura = 100.0 * union.sum() / pixeles_totales
solape_pct = 100.0 * solape.sum() / max(int(union.sum()), 1)
print(f"Cobertura: {cobertura:.1f} %   Solape: {solape_pct:.1f} %")
print(f"Area mediana: {np.median(areas):.0f} px   IoU medio: {np.mean(ious):.3f}   Estabilidad media: {np.mean(estabilidades):.3f}")

In [ ]:
# --- Guardar metricas por mascara (CSV) ---
with open(os.path.join(carpeta_salida, "Mascaras_vit_h_metricas.csv"), "w", newline="") as f:
    escritor = csv.writer(f)
    escritor.writerow(["mascara", "area_px", "predicted_iou", "stability_score"])
    for i, (a, iou, est) in enumerate(zip(areas, ious, estabilidades)):
        escritor.writerow([i, a, round(iou, 4), round(est, 4)])

In [ ]:
# --- Overlay de colores y figura comparativa ---
color_rng = np.random.default_rng(0)
coloreado = imagen_rgb.copy()
for m in mascaras:
    seg = m["segmentation"]
    coloreado[seg] = (0.45 * coloreado[seg] + 0.55 * color_rng.integers(60, 255, size=3)).astype(np.uint8)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(imagen_rgb); ax[0].set_title("Original"); ax[0].axis("off")
ax[1].imshow(coloreado);  ax[1].set_title(f"{len(mascaras)} mascaras (vit_h)"); ax[1].axis("off")
fig.tight_layout()
ruta_figura = os.path.join(carpeta_salida, "Mascaras_vit_h.png")
fig.savefig(ruta_figura, dpi=150, bbox_inches="tight")
plt.show()
print("Guardado:", ruta_figura)